# Lab 8: Full Pipeline -- Kafka -> Spark Streaming -> ML -> Alerts

## Goal

- Build a complete end-to-end real-time pipeline,
- Read transactions from Kafka in Spark,
- Apply ML model scoring in the stream,
- Write alerts back to Kafka.

**Business case (continued):** Time to connect everything. Your system reads live transactions, scores them with the ML model, and pushes alerts downstream.

---

## Part 1: Setup

Start the producer from Lab 1, make sure Kafka is running and the `fraud_model_v2.pkl` from Lab 6 exists.

### Task 1.1 -- Load model in Spark

We'll broadcast the model to all Spark workers.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pickle
import numpy as np

spark = SparkSession.builder.appName("Lab8-Pipeline").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Load sklearn model and broadcast
with open('fraud_model_v2.pkl', 'rb') as f:
    model = pickle.load(f)

model_broadcast = spark.sparkContext.broadcast(model)
print("Model loaded and broadcast")

### Task 1.2 -- Read and parse Kafka stream

Reuse your parsing code from Lab 2.

In [ ]:
tx_schema = StructType([
    StructField("tx_id", StringType()),
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("store", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType()),
])

# YOUR CODE
# Read from Kafka, parse JSON, convert timestamp

---

## Part 2: Apply ML Model in Stream

### Task 2.1 -- UDF for model scoring

Create a Spark UDF that uses the broadcast model to score each row.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

@udf(DoubleType())
def predict_fraud_prob(amount, hour, is_electronics, tx_per_day):
    m = model_broadcast.value
    features = np.array([[amount, hour, is_electronics, tx_per_day]])
    # YOUR CODE
    # Return fraud probability (predict_proba[:, 1])
    pass

### Task 2.2 -- Apply UDF to stream and filter alerts

In [ ]:
# YOUR CODE
# 1. Add feature columns (hour from timestamp, is_electronics flag, etc.)
# 2. Apply predict_fraud_prob UDF
# 3. Filter fraud_probability > 0.5
# 4. Display alerts

### Task 2.3 -- Write alerts to Kafka `alerts` topic

In [ ]:
# YOUR CODE
# Select alert fields, convert to JSON, write to Kafka

---

## Part 3: Windowed Monitoring Dashboard

### Task 3.1 -- Aggregate alerts per 5-minute window

How many alerts per window? What's the average fraud probability?

In [ ]:
# YOUR CODE

---

## Homework

1. Add the rule-based score from Lab 3 alongside the ML score. Compare them in the stream.
2. Write a separate consumer that reads from `alerts` and saves to CSV.
3. Push to Git.

**Next lab:** Dockerize the complete system.

In [ ]:
spark.stop()